# LeetCode 343 — Integer Break bằng Python

Link đề: https://leetcode.com/problems/integer-break/ · Độ khó: **Medium** · Chủ đề: Math (toán), Dynamic Programming (quy hoạch động), Greedy (tham lam)

Skeleton LeetCode:

```python
class Solution:
    def integerBreak(self, n: int) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo hiểu biết về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code (vét cạn mọi cách tách), nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tách số nguyên dương `n` thành **tổng của ít nhất hai số nguyên dương** sao cho **tích** các phần tử là **lớn nhất**; trả về tích lớn nhất đó.
2. Nắm các công cụ Python cần thiết: tích bằng vòng lặp / `math.prod`, `max` nhiều đối số, `divmod`, toán tử `**` và `pow`, danh sách `f = [0] * (n + 1)` làm bảng DP, đệ quy kèm `functools.lru_cache`.
3. Đi từ **vét cạn mọi cách tách** (liệt kê phân hoạch) → **đệ quy có nhớ (top-down)** → **DP bottom-up `O(n²)`** → **chia đều thành `k` phần (AM-GM)** → **công thức toán: dùng càng nhiều số `3` càng tốt (`O(1)`)**.
4. Hiểu mấu chốt: với mỗi phần cuối `j`, phần còn lại `i - j` hoặc **giữ nguyên** hoặc **đã được tách tối ưu** (`f[i - j]`); và hiểu vì sao số `3` là "đơn vị tốt nhất" để tách.
5. Tránh các lỗi kinh điển: quên trường hợp **giữ nguyên** `i - j`, quên các ca nhỏ `n = 2, 3`, xử lý sai dư `1` (phải đổi `3 + 1` thành `2 + 2`).

## 2. Đề bài & ví dụ

Cho một số nguyên `n`. Hãy **tách** `n` thành **tổng của `k` số nguyên dương** (với `k >= 2`) sao cho **tích** của các số đó là **lớn nhất**.

Trả về **tích lớn nhất** có thể đạt được.

**Ví dụ 1**

- Input: `n = 2`
- Output: `1`
- Giải thích: `2 = 1 + 1`, tích `1 × 1 = 1`. (Bắt buộc phải tách thành ít nhất hai số nên không được giữ `2`.)

**Ví dụ 2**

- Input: `n = 10`
- Output: `36`
- Giải thích: `10 = 3 + 3 + 4`, tích `3 × 3 × 4 = 36`.

**Ràng buộc:** `2 <= n <= 58`.

Điểm cần chú ý ngay từ đề: các số sau khi tách phải **dương** (không dùng `0`), và phải có **ít nhất hai** số (`k >= 2`).

In [1]:
examples = [
    (2, 1),
    (10, 36),
    (3, 2),      # 3 = 1 + 2 -> 2
    (4, 4),      # 4 = 2 + 2 -> 4
    (6, 9),      # 6 = 3 + 3 -> 9
]
for n, expected in examples:
    print("n =", n, "-> kỳ vọng", expected)

n = 2 -> kỳ vọng 1
n = 10 -> kỳ vọng 36
n = 3 -> kỳ vọng 2
n = 4 -> kỳ vọng 4
n = 6 -> kỳ vọng 9


## 3. Kiến thức nền

### 3.1. Phân hoạch một số nguyên và tích các phần

Một cách tách `n` thành tổng các số nguyên dương gọi là một **phân hoạch** (partition). Thứ tự các phần **không quan trọng** với tích (`3 × 4 = 4 × 3`), nên ta thường chỉ xét các phần xếp **giảm dần**: `4 = 3 + 1 = 2 + 2 = 2 + 1 + 1 = 1 + 1 + 1 + 1` (ngoài ra còn cách giữ nguyên `4`, nhưng bài này cấm).

Với mỗi phân hoạch, tích là phép nhân tất cả các phần. Python tính tích bằng vòng lặp, hoặc `math.prod` (có từ Python 3.8).

In [2]:
import math

parts = [3, 3, 4]
tich = 1
for p in parts:
    tich *= p
print(sum(parts), tich)                  # 10 36
print(math.prod(parts), math.prod([]))   # 36 1  (tích rỗng = 1)

# các cách tách số 4 thành ít nhất hai phần dương:
for cach in [(3, 1), (2, 2), (2, 1, 1), (1, 1, 1, 1)]:
    print(cach, "tổng", sum(cach), "tích", math.prod(cach))

10 36
36 1
(3, 1) tổng 4 tích 3
(2, 2) tổng 4 tích 4
(2, 1, 1) tổng 4 tích 2
(1, 1, 1, 1) tổng 4 tích 1


### 3.2. `max` với nhiều đối số

`max(a, b, c)` trả về số lớn nhất trong các đối số. Ta sẽ dùng nó để chọn tích tốt nhất giữa nhiều lựa chọn: `max(f[i], f[i - j] * j, (i - j) * j)`.

In [3]:
print(max(2, 7, 5))                      # 7
best = 0
for candidate in [3, 9, 4]:
    best = max(best, candidate)          # cập nhật "tốt nhất đến lúc này"
print(best)                              # 9

7
9


### 3.3. Bảng DP là một `list`

Quy hoạch động (DP) thường lưu đáp án của các bài toán con trong một danh sách: `f[i]` là đáp án cho số `i`. Tạo bảng: `f = [0] * (n + 1)` (có `n + 1` ô, chỉ số `0..n`). Điền theo thứ tự tăng dần của `i` để khi tính `f[i]` thì các `f[nhỏ hơn]` đã sẵn sàng.

In [4]:
n = 5
f = [0] * (n + 1)
f[1] = 1
for i in range(2, n + 1):
    f[i] = f[i - 1] + i                  # ví dụ minh họa quy tắc điền, chưa phải bài này
print(f)                                  # [0, 1, 3, 6, 10, 15]
print(len(f), f[n])                       # 6 15

[0, 1, 3, 6, 10, 15]
6 15


### 3.4. `divmod`, `//`, `%`, `**`, `pow`

- `a // b` là **thương nguyên**, `a % b` là **dư**; `divmod(a, b)` trả về cả hai: `divmod(10, 3) == (3, 1)`.
- `3 ** 4` hoặc `pow(3, 4)` là lũy thừa `3⁴ = 81`. `pow(3, 4, m)` là `3⁴ mod m` (hữu ích khi số quá lớn).
- Python có số nguyên **không giới hạn**, nên `3 ** 18 * 4` không tràn như `int` 32-bit của Java/C++.

In [5]:
print(divmod(10, 3))                      # (3, 1)
q, r = divmod(58, 3)
print(q, r)                               # 19 1  -> 58 = 3*19 + 1
print(3 ** 4, pow(3, 4), pow(3, 4, 10))   # 81 81 1
print(3 ** 18 * 4)                        # 1549681956  (chính là đáp án của n = 58)

(3, 1)
19 1
81 81 1
1549681956


### 3.5. Đệ quy và nhớ kết quả bằng `functools.lru_cache`

Một hàm đệ quy có thể bị gọi lại **nhiều lần với cùng đối số**. `@lru_cache(maxsize=None)` giúp Python **nhớ** kết quả của từng đối số, lần sau trả ngay (đây là **memoization**, DP top-down). Ví dụ Fibonacci: không nhớ thì chạy lâu, có nhớ thì tức thì.

In [6]:
from functools import lru_cache

calls = {"plain": 0, "memo": 0}

def fib_plain(k):
    calls["plain"] += 1
    return k if k < 2 else fib_plain(k - 1) + fib_plain(k - 2)

@lru_cache(maxsize=None)
def fib_memo(k):
    calls["memo"] += 1
    return k if k < 2 else fib_memo(k - 1) + fib_memo(k - 2)

print(fib_plain(20), calls["plain"])     # 6765 21891 lần gọi
print(fib_memo(20), calls["memo"])       # 6765 21 lần gọi

6765 21891
6765 21


### 3.6. Sinh mọi phân hoạch bằng đệ quy (backtracking)

Để **liệt kê** mọi cách tách, ta chọn lần lượt từng phần theo thứ tự **không tăng**: phần tiếp theo luôn `<=` phần trước (để mỗi tập hợp phần chỉ xuất hiện **một lần**, không đếm `3+1` và `1+3` là hai cách). Hàm sinh `gen(con_lai, toi_da)` trả về các danh sách có tổng `con_lai`, mỗi phần `<= toi_da`.

In [7]:
def gen(remaining, max_part):
    """Sinh mọi phân hoạch (parts giảm dần) của remaining, mỗi phần <= max_part."""
    if remaining == 0:
        yield []
        return
    for first in range(min(remaining, max_part), 0, -1):
        for rest in gen(remaining - first, first):
            yield [first] + rest

print(list(gen(5, 5)))
# [[5], [4, 1], [3, 2], [3, 1, 1], [2, 2, 1], [2, 1, 1, 1], [1, 1, 1, 1, 1]]
print(list(gen(5, 4)))                    # cấm phần 5 (tức cấm "giữ nguyên") => ít nhất hai phần

[[5], [4, 1], [3, 2], [3, 1, 1], [2, 2, 1], [2, 1, 1, 1], [1, 1, 1, 1, 1]]
[[4, 1], [3, 2], [3, 1, 1], [2, 2, 1], [2, 1, 1, 1], [1, 1, 1, 1, 1]]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Thử tay với các `n` nhỏ. Với mỗi `n` ta liệt kê các cách tách và lấy tích lớn nhất:

| `n` | cách tách tốt nhất | tích |
|---|---|---|
| 2 | 1 + 1 | 1 |
| 3 | 1 + 2 | 2 |
| 4 | 2 + 2 | 4 |
| 5 | 2 + 3 | 6 |
| 6 | 3 + 3 | 9 |
| 7 | 3 + 4 (= 3 + 2 + 2) | 12 |
| 8 | 3 + 3 + 2 | 18 |
| 9 | 3 + 3 + 3 | 27 |
| 10 | 3 + 3 + 4 (= 3 + 3 + 2 + 2) | 36 |

Quan sát: **không bao giờ có phần `1`** (khi `n >= 4`), phần lớn là `3`, đôi khi có `2` hoặc `4` (tức `2 + 2`). Hãy để máy kiểm tra quan sát này.

In [8]:
def best_partition(n):
    best_p, best_v = None, 0
    for p in gen(n, n - 1):                # n - 1: cấm giữ nguyên n => ít nhất 2 phần
        v = math.prod(p)
        if v > best_v:
            best_p, best_v = p, v
    return best_p, best_v

for n in range(2, 13):
    print(n, best_partition(n))

2 ([1, 1], 1)
3 ([2, 1], 2)
4 ([2, 2], 4)
5 ([3, 2], 6)
6 ([3, 3], 9)
7 ([4, 3], 12)
8 ([3, 3, 2], 18)
9 ([3, 3, 3], 27)
10 ([4, 3, 3], 36)
11 ([3, 3, 3, 2], 54)
12 ([3, 3, 3, 3], 81)


Từ bảng ta rút ra vài **nhận xét trực giác** (sẽ chứng minh ở mục 10):

1. **Tránh phần `1`**: nhân với `1` không làm tích tăng, còn nếu gộp `1` vào một phần `x` khác (đổi `1 + x` thành một phần `x + 1`) thì tích đổi từ `x` thành `x + 1`, lớn hơn.
2. **Phần lớn (`>= 5`) nên tách nhỏ hơn**: `x = 2 + (x - 2)` cho tích `2 * (x - 2) = 2x - 4`, lớn hơn `x` khi `x > 4`. Ví dụ `5 -> 2 * 3 = 6 > 5`.
3. **`4` thì không cần tách thêm**: `4 = 2 + 2` có tích `4 = 4` (hòa).
4. **Giữa `2` và `3`**: `3 + 3 = 6` cho `9`, còn `2 + 2 + 2 = 6` cho `8`. Nên `3` tốt hơn `2`.

In [9]:
# kiểm tra nhanh các nhận xét bằng số
for x in range(5, 12):
    print(f"x = {x}: giữ nguyên {x}, tách 2 + {x - 2} cho {2 * (x - 2)} (tách tốt hơn: {2 * (x - 2) > x})")
print("4 giữ nguyên:", 4, "| 2 + 2:", 2 * 2)
print("2+2+2:", 2 * 2 * 2, "| 3+3:", 3 * 3)

x = 5: giữ nguyên 5, tách 2 + 3 cho 6 (tách tốt hơn: True)
x = 6: giữ nguyên 6, tách 2 + 4 cho 8 (tách tốt hơn: True)
x = 7: giữ nguyên 7, tách 2 + 5 cho 10 (tách tốt hơn: True)
x = 8: giữ nguyên 8, tách 2 + 6 cho 12 (tách tốt hơn: True)
x = 9: giữ nguyên 9, tách 2 + 7 cho 14 (tách tốt hơn: True)
x = 10: giữ nguyên 10, tách 2 + 8 cho 16 (tách tốt hơn: True)
x = 11: giữ nguyên 11, tách 2 + 9 cho 18 (tách tốt hơn: True)
4 giữ nguyên: 4 | 2 + 2: 4
2+2+2: 8 | 3+3: 9


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi cách tách

**Ý tưởng.** Dùng `gen(n, n - 1)` ở mục 3.6 để sinh **mọi** phân hoạch của `n` có ít nhất hai phần (`max_part = n - 1`), tính tích từng phân hoạch rồi lấy `max`. Rõ ràng đúng vì đã thử mọi khả năng.

In [10]:
def integer_break_bruteforce(n: int) -> int:
    best = 0
    for parts in gen(n, n - 1):            # mọi cách tách thành >= 2 phần dương
        best = max(best, math.prod(parts))
    return best


for n, expected in examples:
    print(n, integer_break_bruteforce(n), "(kỳ vọng", expected, ")")

2 1 (kỳ vọng 1 )
10 36 (kỳ vọng 36 )
3 2 (kỳ vọng 2 )
4 4 (kỳ vọng 4 )
6 9 (kỳ vọng 9 )


**Độ phức tạp (Cách 1).** Số phân hoạch của `n`, ký hiệu `p(n)`, tăng rất nhanh (xấp xỉ `e^(π·√(2n/3))`). Mỗi phân hoạch tốn tới `O(n)` để nhân. Time vào cỡ `O(p(n) · n)`, Space `O(n)` cho ngăn xếp đệ quy.

Hãy đếm xem `p(n)` lớn cỡ nào.

In [11]:
import time

for n in [10, 20, 30, 40]:
    t0 = time.time()
    count = sum(1 for _ in gen(n, n - 1))
    print(f"n = {n}: {count} phân hoạch, {time.time() - t0:.3f}s")
# với n = 58 có hơn 700 nghìn phân hoạch, vẫn chạy được nhưng chậm và tốn công vô ích

n = 10: 41 phân hoạch, 0.000s
n = 20: 626 phân hoạch, 0.001s
n = 30: 5603 phân hoạch, 0.008s
n = 40: 37337 phân hoạch, 0.060s


**Nút thắt:** ta liệt kê lại **toàn bộ** các cách tách trong khi nhiều cách chia sẻ cùng một "đuôi". Ví dụ mọi cách tách có đuôi `... + 4` đều cần biết tách tối ưu cho phần còn lại; không cần liệt kê lại từng cái. Ta cần **dùng lại** kết quả tối ưu của số nhỏ hơn.

## 6. Ý tưởng thuật toán / bước đột phá

Gọi `f[i]` là **tích lớn nhất** khi tách `i` thành **ít nhất hai** phần dương. Muốn tính `f[i]`, hãy xét **phần cuối cùng** `j` (với `1 <= j <= i - 1`). Phần còn lại có giá trị `i - j`, và có hai khả năng:

1. **Giữ nguyên `i - j`** (không tách thêm): tích là `(i - j) * j`.
2. **Tách tiếp `i - j`** theo cách tối ưu: tích là `f[i - j] * j`.

Vậy:

```text
f[i] = max over j in [1, i-1] của max( (i - j) * j , f[i - j] * j )
f[1] = 1   (giá trị đệm cho trường hợp "phần còn lại là 1")
```

Đáp án là `f[n]`. Điều then chốt: **bài toán con "tách `i - j` tối ưu" chính là `f[i - j]`**, đã có sẵn khi điền theo thứ tự tăng dần.

Tính tay `f[6]`: `j = 1: max(5*1, f[5]*1 = 6) = 6`; `j = 2: max(4*2 = 8, f[4]*2 = 8) = 8`; `j = 3: max(3*3 = 9, f[3]*3 = 6) = 9`; `j = 4: max(2*4 = 8, f[2]*4 = 4)`; `j = 5: max(1*5, f[1]*5)`; lớn nhất là `9`.

In [12]:
f = {1: 1, 2: 1, 3: 2, 4: 4, 5: 6}           # các giá trị đã biết để tính tay f[6]
i = 6
for j in range(1, i):
    giu_nguyen = (i - j) * j
    tach_tiep = f[i - j] * j
    print(f"j={j}: giữ nguyên ({i - j})*{j} = {giu_nguyen}, tách tiếp f[{i - j}]*{j} = {tach_tiep}")
print("f[6] =", max(max((i - j) * j, f[i - j] * j) for j in range(1, i)))     # 9

j=1: giữ nguyên (5)*1 = 5, tách tiếp f[5]*1 = 6
j=2: giữ nguyên (4)*2 = 8, tách tiếp f[4]*2 = 8
j=3: giữ nguyên (3)*3 = 9, tách tiếp f[3]*3 = 6
j=4: giữ nguyên (2)*4 = 8, tách tiếp f[2]*4 = 4
j=5: giữ nguyên (1)*5 = 5, tách tiếp f[1]*5 = 5
f[6] = 9


## 7. Các cách tiếp theo

### Cách 2 — Đệ quy có nhớ (top-down, memoization)

**Giải thích từng đoạn.**

1. `best(i)` = tích lớn nhất khi tách `i` thành ít nhất hai phần.
2. Với mỗi phần cuối `j` từ `1` đến `i - 1`, thử `j * (i - j)` (giữ nguyên) và `j * best(i - j)` (tách tiếp), lấy `max`.
3. Trường hợp cơ sở: `best(1) = 1`. Dùng `@lru_cache` để mỗi `i` chỉ tính **một lần**.
4. Gọi `best(n)`.

In [13]:
def integer_break_memo(n: int) -> int:
    @lru_cache(maxsize=None)
    def best(i: int) -> int:
        if i == 1:
            return 1
        result = 0
        for j in range(1, i):
            result = max(result, j * (i - j), j * best(i - j))
        return result

    return best(n)


for n, expected in examples:
    print(n, integer_break_memo(n), "(kỳ vọng", expected, ")")

2 1 (kỳ vọng 1 )
10 36 (kỳ vọng 36 )
3 2 (kỳ vọng 2 )
4 4 (kỳ vọng 4 )
6 9 (kỳ vọng 9 )


In [14]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
from functools import lru_cache

class SolutionMemo:
    def integerBreak(self, n: int) -> int:
        @lru_cache(maxsize=None)
        def best(i: int) -> int:
            if i == 1:
                return 1
            result = 0
            for j in range(1, i):
                result = max(result, j * (i - j), j * best(i - j))
            return result

        return best(n)


print(SolutionMemo().integerBreak(10))      # 36

36


**Độ phức tạp Cách 2:** có `n` trạng thái `i`, mỗi trạng thái lặp `O(i)` ⇒ **Time `O(n²)`**; bộ nhớ cho bảng nhớ và ngăn xếp đệ quy: **Space `O(n)`**.

### Cách 3 — DP bottom-up (điền bảng từ nhỏ đến lớn)

**Giải thích từng đoạn.**

1. `f = [1] * (n + 1)`: bảng `f[0..n]`; ta chỉ dùng `f[1..n]`, khởi tạo `1` cho an toàn (`f[1] = 1`).
2. Vòng ngoài `i` từ `2` đến `n`: tính `f[i]`.
3. Vòng trong `j` từ `1` đến `i - 1`: `f[i] = max(f[i], f[i - j] * j, (i - j) * j)`.
4. Trả về `f[n]`.

In [15]:
def integer_break_dp(n: int) -> int:
    f = [1] * (n + 1)                       # f[i]: tích lớn nhất khi tách i (>= 2 phần); f[1] = 1
    for i in range(2, n + 1):
        f[i] = 0                            # đặt lại 0 để không "mượn" giá trị khởi tạo 1
        for j in range(1, i):
            f[i] = max(f[i], f[i - j] * j, (i - j) * j)
    return f[n]


for n, expected in examples:
    print(n, integer_break_dp(n), "(kỳ vọng", expected, ")")

2 1 (kỳ vọng 1 )
10 36 (kỳ vọng 36 )
3 2 (kỳ vọng 2 )
4 4 (kỳ vọng 4 )
6 9 (kỳ vọng 9 )


In [16]:
# Code nộp LeetCode cho Cách 3
class SolutionDP:
    def integerBreak(self, n: int) -> int:
        f = [1] * (n + 1)
        for i in range(2, n + 1):
            f[i] = 0
            for j in range(1, i):
                f[i] = max(f[i], f[i - j] * j, (i - j) * j)
        return f[n]


print(SolutionDP().integerBreak(10))        # 36

36


**Độ phức tạp Cách 3:** hai vòng lồng ⇒ **Time `O(n²)`** (với `n <= 58` chỉ cỡ `1600` bước), **Space `O(n)`** cho bảng `f`.

### Cách 4 — Chia đều thành đúng `k` phần (bất đẳng thức AM-GM)

**Ý tưởng.** Nếu đã **cố định số phần `k`**, tích lớn nhất đạt được khi các phần **chênh nhau không quá `1`**. Lý do: nếu có hai phần `a >= b + 2` thì thay bằng `a - 1` và `b + 1` được `(a - 1)(b + 1) = ab + a - b - 1 > ab`, tích tăng. Vậy tối ưu cho `k` phần là: `q = n // k`, `r = n % k`; có `r` phần bằng `q + 1` và `k - r` phần bằng `q`; tích `(q + 1)^r · q^(k - r)`. Cuối cùng thử **mọi `k` từ `2` đến `n`** và lấy `max`.

Ví dụ `n = 10`, `k = 3`: `q = 3`, `r = 1` ⇒ phần `4, 3, 3` ⇒ tích `36`.

In [17]:
n = 10
for k in range(2, n + 1):
    q, r = divmod(n, k)
    print(f"k={k}: q={q}, r={r}, các phần: {r} số {q + 1} và {k - r} số {q}, tích = {(q + 1) ** r * q ** (k - r)}")

k=2: q=5, r=0, các phần: 0 số 6 và 2 số 5, tích = 25
k=3: q=3, r=1, các phần: 1 số 4 và 2 số 3, tích = 36
k=4: q=2, r=2, các phần: 2 số 3 và 2 số 2, tích = 36
k=5: q=2, r=0, các phần: 0 số 3 và 5 số 2, tích = 32
k=6: q=1, r=4, các phần: 4 số 2 và 2 số 1, tích = 16
k=7: q=1, r=3, các phần: 3 số 2 và 4 số 1, tích = 8
k=8: q=1, r=2, các phần: 2 số 2 và 6 số 1, tích = 4
k=9: q=1, r=1, các phần: 1 số 2 và 8 số 1, tích = 2
k=10: q=1, r=0, các phần: 0 số 2 và 10 số 1, tích = 1


In [18]:
def integer_break_equal_parts(n: int) -> int:
    best = 0
    for k in range(2, n + 1):               # k = số phần
        q, r = divmod(n, k)
        best = max(best, (q + 1) ** r * q ** (k - r))
    return best


for n, expected in examples:
    print(n, integer_break_equal_parts(n), "(kỳ vọng", expected, ")")

2 1 (kỳ vọng 1 )
10 36 (kỳ vọng 36 )
3 2 (kỳ vọng 2 )
4 4 (kỳ vọng 4 )
6 9 (kỳ vọng 9 )


In [19]:
# Code nộp LeetCode cho Cách 4
class SolutionEqual:
    def integerBreak(self, n: int) -> int:
        best = 0
        for k in range(2, n + 1):
            q, r = divmod(n, k)
            best = max(best, (q + 1) ** r * q ** (k - r))
        return best


print(SolutionEqual().integerBreak(10))     # 36

36


**Độ phức tạp Cách 4:** `n - 1` giá trị `k`, mỗi lần lũy thừa số lớn tốn thêm chút (coi như `O(1)` với `n <= 58`) ⇒ **Time `O(n)`** (về chặt chẽ hơn là `O(n · log n)` phép nhân), **Space `O(1)`**.

### Cách 5 — Công thức toán: càng nhiều số `3` càng tốt (`O(1)`)

**Ý tưởng.** Hàm `(n / k)^k` đạt cực đại khi `n / k ≈ e ≈ 2,718`, nên phần tối ưu nên gần `e`, tức là **`3`** (hoặc `2`). Cụ thể, với `n >= 5`:

- Mọi phần `>= 5` nên tách tiếp (vì `2(x - 2) > x`) ⇒ các phần chỉ là `1, 2, 3, 4`.
- Phần `1` có hại, `4 = 2 + 2` nên coi như hai phần `2`.
- Có từ **ba** phần `2` trở lên thì nên đổi `2 + 2 + 2` thành `3 + 3` (tích `9 > 8`) ⇒ **tối đa hai phần `2`**.

Vậy `n` được tách thành các số `3` và (nếu cần) một hoặc hai số `2`:

- `n % 3 == 0`: toàn số `3` ⇒ `3^(n/3)`.
- `n % 3 == 1`: `n = 3(m - 1) + 4` ⇒ `3^(m - 1) · 4` với `m = n // 3` (đổi `3 + 1` thành `2 + 2`).
- `n % 3 == 2`: `n = 3m + 2` ⇒ `3^m · 2`.
- Trường hợp `n < 4`: `n - 1` (vì bắt buộc tách: `2 -> 1 + 1 = 1`, `3 -> 1 + 2 = 2`).

In [20]:
# kiểm tra các "bổ đề" bằng số
print([x for x in range(1, 12) if 2 * (x - 2) > x])          # phần x nên tách: 5, 6, 7, ... (x >= 5)
print("2+2+2 =", 2 ** 3, "< 3+3 =", 3 ** 2)
print("3 + 1 -> tích", 3 * 1, "| 2 + 2 -> tích", 2 * 2)
for n in [6, 7, 8]:
    print(n, "chia 3 dư", n % 3, "->", n // 3, "số 3")

[5, 6, 7, 8, 9, 10, 11]
2+2+2 = 8 < 3+3 = 9
3 + 1 -> tích 3 | 2 + 2 -> tích 4
6 chia 3 dư 0 -> 2 số 3
7 chia 3 dư 1 -> 2 số 3
8 chia 3 dư 2 -> 2 số 3


In [21]:
def integer_break_math(n: int) -> int:
    if n < 4:
        return n - 1
    q, r = divmod(n, 3)
    if r == 0:
        return 3 ** q
    if r == 1:
        return 3 ** (q - 1) * 4             # lấy bớt một số 3, ghép với số 1 thành 4 = 2 + 2
    return 3 ** q * 2


for n, expected in examples:
    print(n, integer_break_math(n), "(kỳ vọng", expected, ")")

2 1 (kỳ vọng 1 )
10 36 (kỳ vọng 36 )
3 2 (kỳ vọng 2 )
4 4 (kỳ vọng 4 )
6 9 (kỳ vọng 9 )


In [22]:
# Code nộp LeetCode cho Cách 5
class SolutionMath:
    def integerBreak(self, n: int) -> int:
        if n < 4:
            return n - 1
        q, r = divmod(n, 3)
        if r == 0:
            return 3 ** q
        if r == 1:
            return 3 ** (q - 1) * 4
        return 3 ** q * 2


print(SolutionMath().integerBreak(10), SolutionMath().integerBreak(58))      # 36 1549681956

36 1549681956


**Độ phức tạp Cách 5:** một lũy thừa duy nhất ⇒ **Time `O(1)`** (coi lũy thừa là hằng với `n <= 58`; chặt chẽ là `O(log n)` phép nhân nếu dùng bình phương liên tiếp), **Space `O(1)`**.

## 8. TRACE từng bước

Trace DP (Cách 3) cho `n = 6` (in cả giá trị khi giữ nguyên lẫn khi tách tiếp), rồi trace công thức toán (Cách 5) cho `n = 10` và `n = 7`.

In [23]:
def trace_dp(n: int) -> int:
    f = [1] * (n + 1)
    print(f"n = {n}, bắt đầu f = {f}")
    for i in range(2, n + 1):
        f[i] = 0
        print(f"--- i = {i}")
        for j in range(1, i):
            giu = (i - j) * j
            tiep = f[i - j] * j
            cu = f[i]
            f[i] = max(f[i], giu, tiep)
            print(f"    j={j}: giữ {i - j}*{j}={giu}, tách tiếp f[{i - j}]*{j}={tiep} | f[{i}]: {cu} -> {f[i]}")
        print(f"    => f[{i}] = {f[i]}")
    return f[n]


assert trace_dp(6) == 9

n = 6, bắt đầu f = [1, 1, 1, 1, 1, 1, 1]
--- i = 2
    j=1: giữ 1*1=1, tách tiếp f[1]*1=1 | f[2]: 0 -> 1
    => f[2] = 1
--- i = 3
    j=1: giữ 2*1=2, tách tiếp f[2]*1=1 | f[3]: 0 -> 2
    j=2: giữ 1*2=2, tách tiếp f[1]*2=2 | f[3]: 2 -> 2
    => f[3] = 2
--- i = 4
    j=1: giữ 3*1=3, tách tiếp f[3]*1=2 | f[4]: 0 -> 3
    j=2: giữ 2*2=4, tách tiếp f[2]*2=2 | f[4]: 3 -> 4
    j=3: giữ 1*3=3, tách tiếp f[1]*3=3 | f[4]: 4 -> 4
    => f[4] = 4
--- i = 5
    j=1: giữ 4*1=4, tách tiếp f[4]*1=4 | f[5]: 0 -> 4
    j=2: giữ 3*2=6, tách tiếp f[3]*2=4 | f[5]: 4 -> 6
    j=3: giữ 2*3=6, tách tiếp f[2]*3=3 | f[5]: 6 -> 6
    j=4: giữ 1*4=4, tách tiếp f[1]*4=4 | f[5]: 6 -> 6
    => f[5] = 6
--- i = 6
    j=1: giữ 5*1=5, tách tiếp f[5]*1=6 | f[6]: 0 -> 6
    j=2: giữ 4*2=8, tách tiếp f[4]*2=8 | f[6]: 6 -> 8
    j=3: giữ 3*3=9, tách tiếp f[3]*3=6 | f[6]: 8 -> 9
    j=4: giữ 2*4=8, tách tiếp f[2]*4=4 | f[6]: 9 -> 9
    j=5: giữ 1*5=5, tách tiếp f[1]*5=5 | f[6]: 9 -> 9
    => f[6] = 9


In [24]:
def trace_math(n: int) -> int:
    print(f"n = {n}")
    if n < 4:
        print(f"  n < 4 nên trả n - 1 = {n - 1}")
        return n - 1
    q, r = divmod(n, 3)
    print(f"  divmod({n}, 3) = (q={q}, r={r})")
    if r == 0:
        ans = 3 ** q
        print(f"  dư 0: toàn số 3 -> 3^{q} = {ans}")
    elif r == 1:
        ans = 3 ** (q - 1) * 4
        print(f"  dư 1: dùng {q - 1} số 3 và một số 4 -> 3^{q - 1} * 4 = {ans}")
    else:
        ans = 3 ** q * 2
        print(f"  dư 2: dùng {q} số 3 và một số 2 -> 3^{q} * 2 = {ans}")
    return ans


assert trace_math(10) == 36
assert trace_math(7) == 12
assert trace_math(9) == 27
assert trace_math(2) == 1

n = 10
  divmod(10, 3) = (q=3, r=1)
  dư 1: dùng 2 số 3 và một số 4 -> 3^2 * 4 = 36
n = 7
  divmod(7, 3) = (q=2, r=1)
  dư 1: dùng 1 số 3 và một số 4 -> 3^1 * 4 = 12
n = 9
  divmod(9, 3) = (q=3, r=0)
  dư 0: toàn số 3 -> 3^3 = 27
n = 2
  n < 4 nên trả n - 1 = 1


Bảng trace DP `n = 6` (mỗi dòng là `f[i]` sau khi xét hết `j`):

| `i` | các ứng viên `max((i-j)*j, f[i-j]*j)` theo `j = 1..i-1` | `f[i]` |
|---|---|---|
| 2 | j=1: max(1, 1) = 1 | 1 |
| 3 | j=1: max(2, 1) = 2; j=2: max(2, 2) = 2 | 2 |
| 4 | j=1: 3; j=2: max(4, 2·2) = 4; j=3: max(3, 1·3) = 3 | 4 |
| 5 | j=1: max(4, f[4]·1 = 4) = 4; j=2: max(6, f[3]·2 = 4) = 6; j=3: max(6, f[2]·3 = 3) = 6; j=4: max(4, 4) | 6 |
| 6 | j=3: max(9, f[3]·3=6)=9 | 9 |

Bảng trace công thức `n = 10`: `10 // 3 = 3`, dư `1` ⇒ dùng `3^(3 - 1) · 4 = 9 · 4 = 36`, tương ứng cách tách `3 + 3 + 4`.

## 9. Code nộp LeetCode

Bản nộp dùng **Cách 5 (công thức toán)**: ngắn, `O(1)`, và đã được đối chiếu với DP cho mọi `n` từ `2` đến `58`. (Nếu bạn chưa tự tin chứng minh công thức, nộp bản DP `O(n²)` ở Cách 3 cũng đạt vì `n <= 58`.)

In [25]:
class Solution:
    def integerBreak(self, n: int) -> int:
        if n < 4:                              # bắt buộc tách thành >= 2 phần: 2 -> 1, 3 -> 2
            return n - 1
        q, r = divmod(n, 3)
        if r == 0:                             # toàn số 3
            return 3 ** q
        if r == 1:                             # 3 + 1 đổi thành 2 + 2
            return 3 ** (q - 1) * 4
        return 3 ** q * 2                      # r == 2: các số 3 và một số 2


sol = Solution()
for n, expected in examples:
    print(n, "->", sol.integerBreak(n), "(kỳ vọng", expected, ")")

2 -> 1 (kỳ vọng 1 )
10 -> 36 (kỳ vọng 36 )
3 -> 2 (kỳ vọng 2 )
4 -> 4 (kỳ vọng 4 )
6 -> 9 (kỳ vọng 9 )


## 10. Vì sao thuật toán đúng

**DP (Cách 3).**

- `f[i]` là tích lớn nhất khi tách `i` thành **ít nhất hai** phần. Xét một cách tách tối ưu bất kỳ và gọi `j` là một phần của nó (sắp lại để `j` là phần cuối). Phần còn lại tổng `i - j`: hoặc nó là **một** phần (tích `(i - j) * j`), hoặc là **nhiều** phần, và khi đó tích các phần đó không thể vượt `f[i - j]` (nếu vượt thì thay bằng cách tối ưu cho `i - j` sẽ tốt hơn, mâu thuẫn). Vậy `f[i]` đúng bằng `max` của hai khả năng trên theo mọi `j`.
- `f[1] = 1` chỉ dùng như "tích của phần còn lại là 1" khi `i - j = 1`; trường hợp đó `(i - j) * j` cũng bằng `j`, nhất quán.

**Công thức (Cách 5).** Gọi một cách tách tối ưu cho `n >= 5` có các phần `a₁, ..., a_k`.

1. **Không có phần `1`**: nếu có phần `1`, ghép với phần `x` khác (tồn tại vì `n >= 3`) thành `x + 1`: tích đổi từ `1 · x = x` thành `x + 1 > x`.
2. **Không có phần `x >= 5`**: tách `x = 2 + (x - 2)` được `2(x - 2) = 2x - 4 > x` khi `x > 4`.
3. Vậy các phần thuộc `{2, 3, 4}`. Phần `4` thay bằng `2 + 2` (cùng tích `4`), nên có thể coi phần chỉ là `2` hoặc `3`.
4. **Có tối đa hai phần `2`**: ba phần `2` (tổng `6`, tích `8`) thay bằng hai phần `3` (tổng `6`, tích `9`).
5. Vậy `n = 3a + 2b` với `b ∈ {0, 1, 2}`: `n % 3 == 0 -> b = 0`; `n % 3 == 1 -> b = 2` (`3(q-1) + 4`); `n % 3 == 2 -> b = 1`. Đây đúng là công thức. `n = 2, 3` vì bắt buộc tách nên là `1` và `2`; `n = 4` ra `4` đúng công thức (`q = 1, r = 1 -> 3^0 · 4 = 4`).

**Chia đều (Cách 4).** Với `k` phần cố định, hoán đổi `(a, b) -> (a - 1, b + 1)` khi `a >= b + 2` làm tích tăng nghiêm ngặt, nên tối ưu có các phần chênh nhau `<= 1`; thử mọi `k` nên không sót.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê mọi phân hoạch | cỡ `O(p(n) · n)` (rất lớn) | `O(n)` |
| 2. Đệ quy có nhớ | `O(n²)` | `O(n)` |
| 3. DP bottom-up | `O(n²)` | `O(n)` |
| 4. Chia đều thành `k` phần | `O(n)` phép tính lũy thừa | `O(1)` |
| 5. Công thức số `3` | `O(1)` (một lũy thừa) | `O(1)` |

Với `n <= 58` cả DP và công thức đều tức thì; nhưng nếu đề cho `n` lớn (ví dụ `10^9` kèm modulo) thì chỉ công thức dùng `pow(3, e, mod)` còn chạy nổi.

## 12. Bộ test

Gom các cách vào `impls` rồi so sánh. Cách 1 (vét cạn) chỉ chạy được với `n` nhỏ (`<= 30`), nên ta tách ra khi `n` lớn.

In [26]:
sol = Solution()
impls = {
    "memo": integer_break_memo,
    "dp": integer_break_dp,
    "equal": integer_break_equal_parts,
    "math": integer_break_math,
    "leetcode": sol.integerBreak,
    "submit_memo": SolutionMemo().integerBreak,
    "submit_dp": SolutionDP().integerBreak,
    "submit_equal": SolutionEqual().integerBreak,
    "submit_math": SolutionMath().integerBreak,
}


def check_all(n, expected, label, with_brute=True):
    for key, fn in impls.items():
        actual = fn(n)
        assert actual == expected, (label, key, n, actual, expected)
    if with_brute:
        assert integer_break_bruteforce(n) == expected, (label, "brute", n)
    print("PASS", label, "n =", n, "->", expected)


for n, expected in examples:
    check_all(n, expected, "ví dụ đề")

PASS ví dụ đề n = 2 -> 1
PASS ví dụ đề n = 10 -> 36
PASS ví dụ đề n = 3 -> 2
PASS ví dụ đề n = 4 -> 4
PASS ví dụ đề n = 6 -> 9


### 12.1. Các trường hợp biên

Các giá trị nhỏ nhất (`2`, `3`, `4`), từng dư `0/1/2` khi chia `3`, và giá trị lớn nhất `58`.

In [27]:
check_all(2, 1, "nhỏ nhất: 1 + 1")
check_all(3, 2, "3 = 1 + 2")
check_all(4, 4, "4 = 2 + 2 (hòa với giữ nguyên 4, nhưng bị cấm giữ nguyên)")
check_all(5, 6, "5 = 2 + 3")
check_all(7, 12, "7 = 3 + 4")
check_all(8, 18, "8 = 3 + 3 + 2")
check_all(9, 27, "9 = 3 + 3 + 3")
check_all(11, 54, "11 = 3 + 3 + 3 + 2")
check_all(12, 81, "12 = 3 * 4")
check_all(57, 3 ** 19, "57 chia hết cho 3", with_brute=False)
check_all(58, 3 ** 18 * 4, "lớn nhất: 58 = 3*18 + 4", with_brute=False)

PASS nhỏ nhất: 1 + 1 n = 2 -> 1
PASS 3 = 1 + 2 n = 3 -> 2
PASS 4 = 2 + 2 (hòa với giữ nguyên 4, nhưng bị cấm giữ nguyên) n = 4 -> 4
PASS 5 = 2 + 3 n = 5 -> 6
PASS 7 = 3 + 4 n = 7 -> 12
PASS 8 = 3 + 3 + 2 n = 8 -> 18
PASS 9 = 3 + 3 + 3 n = 9 -> 27
PASS 11 = 3 + 3 + 3 + 2 n = 11 -> 54
PASS 12 = 3 * 4 n = 12 -> 81
PASS 57 chia hết cho 3 n = 57 -> 1162261467
PASS lớn nhất: 58 = 3*18 + 4 n = 58 -> 1549681956


### 12.2. Phản ví dụ cho các cách làm sai

Ba "ý tưởng" nghe hợp lý nhưng sai: (a) **chia đôi** `n` (`n // 2` và phần còn lại); (b) **chỉ tách thành hai phần** gần nhau; (c) **tham lam toàn số `2`**.

In [28]:
def wrong_half(n):
    return (n // 2) * (n - n // 2)                 # SAI: chỉ chia đôi


def wrong_two_parts(n):
    return max(j * (n - j) for j in range(1, n))   # SAI: chỉ cho phép đúng hai phần


def wrong_all_twos(n):
    return 2 ** (n // 2) if n % 2 == 0 else 2 ** (n // 2 - 1) * 3     # SAI: ưu tiên số 2


for n in [6, 10, 12]:
    print(n, "đúng:", sol.integerBreak(n), "| nửa:", wrong_half(n), "| hai phần:", wrong_two_parts(n), "| toàn 2:", wrong_all_twos(n))
assert wrong_half(10) != sol.integerBreak(10)
assert wrong_two_parts(10) != sol.integerBreak(10)
assert wrong_all_twos(12) != sol.integerBreak(12)

6 đúng: 9 | nửa: 9 | hai phần: 9 | toàn 2: 8
10 đúng: 36 | nửa: 25 | hai phần: 25 | toàn 2: 32
12 đúng: 81 | nửa: 36 | hai phần: 36 | toàn 2: 64


### 12.3. Đối chiếu mọi `n` hợp lệ và ngẫu nhiên có seed

Vì miền `n` chỉ gồm `57` giá trị (`2..58`), ta **kiểm tra toàn bộ** miền bằng một oracle độc lập: vét cạn `gen` cho `n <= 30` và DP cho `n` còn lại. Thêm một lượt ngẫu nhiên có seed để kiểm tra đủ mọi hàm.

In [29]:
import random

oracle = {n: integer_break_bruteforce(n) for n in range(2, 31)}      # vét cạn các n nhỏ
for n in range(2, 59):
    ref = integer_break_dp(n)
    if n in oracle:
        assert oracle[n] == ref, (n, oracle[n], ref)                  # DP khớp vét cạn
    for key, fn in impls.items():
        assert fn(n) == ref, (key, n, fn(n), ref)
print("PASS: mọi n từ 2 đến 58 khớp (vét cạn cho n <= 30)")

rng = random.Random(343)
for _ in range(500):
    n = rng.randint(2, 58)
    ref = integer_break_dp(n)
    assert all(fn(n) == ref for fn in impls.values()), n
print("PASS: 500 lượt ngẫu nhiên")

# nhắc lại: kết quả lớn nhất trong miền đề vẫn nhỏ hơn 2**31
assert max(integer_break_dp(n) for n in range(2, 59)) < 2 ** 31
print("đáp án lớn nhất:", max(integer_break_dp(n) for n in range(2, 59)))

PASS: mọi n từ 2 đến 58 khớp (vét cạn cho n <= 30)


PASS: 500 lượt ngẫu nhiên
đáp án lớn nhất: 1549681956


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ dùng `f[i - j] * j`, quên trường hợp giữ nguyên `i - j`

In [30]:
def sai_quen_giu_nguyen(n):
    f = [1] * (n + 1)
    for i in range(2, n + 1):
        f[i] = 0
        for j in range(1, i):
            f[i] = max(f[i], f[i - j] * j)           # SAI: thiếu (i - j) * j
    return f[n]


print("sai:", [sai_quen_giu_nguyen(n) for n in range(2, 8)])
print("đúng:", [Solution().integerBreak(n) for n in range(2, 8)])

sai: [1, 2, 3, 4, 6, 9]
đúng: [1, 2, 4, 6, 9, 12]


**Vì sao sai:** `f[i - j]` bắt buộc tách `i - j` thành **ít nhất hai** phần, nhưng đôi khi **giữ nguyên** `i - j` lại tốt hơn (ví dụ `i - j = 4` giữ `4` tốt hơn tách; hay `i - j = 3` giữ nguyên `3` hơn tách `1 + 2`). Bỏ trường hợp đó nên `f[4]` ra `3` thay vì `4`. **Cách sửa:** `max(f[i], f[i - j] * j, (i - j) * j)`.

### Lỗi 2: Công thức `3^q` quên các ca `n < 4`

In [31]:
def sai_quen_ca_nho(n):
    q, r = divmod(n, 3)
    if r == 0:
        return 3 ** q
    if r == 1:
        return 3 ** (q - 1) * 4
    return 3 ** q * 2


print("sai n=2:", sai_quen_ca_nho(2), "| đúng:", Solution().integerBreak(2))
print("sai n=3:", sai_quen_ca_nho(3), "| đúng:", Solution().integerBreak(3))

sai n=2: 2 | đúng: 1
sai n=3: 3 | đúng: 2


**Vì sao sai:** `n = 2` (dư `2`, `q = 0`) cho `3^0 · 2 = 2`, nhưng không được giữ `2` nguyên (cần `k >= 2`); `n = 3` cho `3`, cũng bị cấm giữ nguyên, đáp án là `2`. **Cách sửa:** xử lý riêng `n < 4` trả `n - 1`.

### Lỗi 3: Dư `1` bị xử lý là "thêm phần 1"

In [32]:
def sai_du_mot(n):
    if n < 4:
        return n - 1
    q, r = divmod(n, 3)
    if r == 0:
        return 3 ** q
    if r == 1:
        return 3 ** q * 1                  # SAI: phần dư 1 nhân với 1 không giúp gì
    return 3 ** q * 2


print("sai n=7:", sai_du_mot(7), "| đúng:", Solution().integerBreak(7))      # 7 = 3+3+1 (9) so với 3+2+2 (12)
print("sai n=10:", sai_du_mot(10), "| đúng:", Solution().integerBreak(10))

sai n=7: 9 | đúng: 12
sai n=10: 27 | đúng: 36


**Vì sao sai:** khi dư `1`, `3 + 1 = 4` nên tách `2 + 2` cho tích `4` thay vì `3 · 1 = 3`. **Cách sửa:** lấy bớt một số `3` và nhân với `4`: `3^(q - 1) * 4`.

### Lỗi 4: Cho phép giữ nguyên `n` (không bắt buộc `k >= 2`)

In [33]:
def sai_cho_giu_nguyen(n):
    g = [1] * (n + 1)                          # g[i]: tích lớn nhất của MỌI phân hoạch, kể cả giữ nguyên i
    for i in range(1, n + 1):
        g[i] = i                               # SAI cho đề này: cho phép giữ nguyên chính n
        for j in range(1, i):
            g[i] = max(g[i], g[i - j] * j)
    return g[n]


print("sai:", [sai_cho_giu_nguyen(n) for n in (2, 3, 4)], "| đúng:", [Solution().integerBreak(n) for n in (2, 3, 4)])

sai: [2, 3, 4] | đúng: [1, 2, 4]


**Vì sao sai:** `g[2] = 2` vì cho phép giữ `2`, trong khi đề buộc tách thành **ít nhất hai** số nên `n = 2` phải ra `1`. Đây là lý do `f[1] = 1` và `f[i]` chỉ xét `j < i`. **Cách sửa:** chỉ cho giữ nguyên với phần **con** (`(i - j) * j`), không cho giữ nguyên chính `i`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tách thành các phần, **tối đa hóa tích**" ⇒ nghĩ ngay tới **số `3`** (gần `e`), rồi `2`; tránh `1`; `4` = `2 + 2`.
- "Tách thành các phần, **tối đa hóa tổng/giá trị**, mỗi phần có thể giữ hoặc tách tiếp" ⇒ DP kiểu `f[i] = max(giữ, f[i - j] ...)` (cũng là mẫu của **Cutting Rope**, **Coin Change** kiểu cộng dồn).
- Khi miền nhỏ (`n <= 58`) hãy **vét cạn** để tìm quy luật rồi mới chứng minh công thức; vét cạn cũng là oracle kiểm tra cuối cùng.
- `divmod(n, 3)` gọn hơn viết riêng `n // 3` và `n % 3`.
- Nhân trong Python không tràn số nên không cần lo `int` 32-bit; nhưng nếu đề yêu cầu `mod 10^9 + 7` hãy dùng `pow(3, e, MOD)`.
- Luôn thử `n = 2, 3, 4` trước khi nộp: đó là các ca mà công thức tổng quát hay hỏng.

## 15. Biến thể của bài

1. **Trả về chính cách tách** (danh sách các phần), không chỉ tích.
2. **Cutting Rope II** (Lĩnh Khấu): `n` lên tới `10^9`, trả tích theo `mod 10^9 + 7` ⇒ dùng công thức `3` với `pow(3, e, MOD)`.
3. **Tách đúng `k` phần**: đáp án là `(q + 1)^r · q^(k - r)` với `q, r = divmod(n, k)` (chính là một bước của Cách 4).
4. **Tối đa hóa tổng bình phương** hoặc các hàm khác: không còn quy luật "số `3`", quay về DP.
5. **Maximize Number of Nice Divisors (1808)**: cùng bản chất, tách số thừa số nguyên tố để tối đa hóa tích; **Perfect Squares (279)** và **Coin Change (322)** là họ DP "tách tổng thành các phần".

In [34]:
MOD = 10 ** 9 + 7


def integer_break_parts(n):
    """Trả về (tích, danh sách các phần) bằng công thức số 3."""
    if n < 4:
        return n - 1, ([1, 1] if n == 2 else [1, 2])
    q, r = divmod(n, 3)
    if r == 0:
        parts = [3] * q
    elif r == 1:
        parts = [3] * (q - 1) + [2, 2]
    else:
        parts = [3] * q + [2]
    return math.prod(parts), parts


def cutting_rope_mod(n):
    """Biến thể n lớn (đến 1e9), trả về tích mod 1e9+7 (giả sử n >= 2)."""
    if n < 4:
        return n - 1
    q, r = divmod(n, 3)
    if r == 0:
        return pow(3, q, MOD)
    if r == 1:
        return pow(3, q - 1, MOD) * 4 % MOD
    return pow(3, q, MOD) * 2 % MOD


def best_exact_k(n, k):
    """Tích lớn nhất khi tách n thành đúng k phần dương (k <= n)."""
    q, r = divmod(n, k)
    return (q + 1) ** r * q ** (k - r)


print(integer_break_parts(10))                # (36, [3, 3, 2, 2])
print(integer_break_parts(2), integer_break_parts(3))
print(cutting_rope_mod(10), cutting_rope_mod(10 ** 9))
print(best_exact_k(10, 3), best_exact_k(10, 2), best_exact_k(10, 5))     # 36 25 32
assert cutting_rope_mod(58) == 3 ** 18 * 4 % MOD
for n in range(2, 20):
    prod, parts = integer_break_parts(n)
    assert sum(parts) == n and len(parts) >= 2 and math.prod(parts) == prod == Solution().integerBreak(n), n

(36, [3, 3, 2, 2])
(1, [1, 1]) (2, [1, 2])
36 897135186
36 25 32


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê phân hoạch | thử mọi cách tách | rất lớn | `O(n)` | oracle kiểm tra `n` nhỏ |
| 2. Đệ quy có nhớ | `best(i)` + `lru_cache` | `O(n²)` | `O(n)` | khi quen top-down |
| 3. DP bottom-up | `f[i] = max(f[i-j]*j, (i-j)*j)` | `O(n²)` | `O(n)` | an toàn, không cần chứng minh toán |
| 4. Chia đều `k` phần | AM-GM, thử mọi `k` | `O(n)` | `O(1)` | khi cần giải thích bằng bất đẳng thức |
| 5. Công thức số `3` | `3^q` và xử lý dư | `O(1)` | `O(1)` | **nên nộp** (và cho `n` lớn với `mod`) |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `divmod(17, 5)`, `max(4, 2 * 3, 1 * 5)`, `3 ** 3 * 2`, `Solution().integerBreak(6)`, `Solution().integerBreak(11)`.

**Bài 2 (tính tay).** Điền bảng `f[1..7]` theo công thức `f[i] = max over j của max((i - j) * j, f[i - j] * j)` với `f[1] = 1`, rồi so với công thức số `3` cho `n = 7`.

**Bài 3 (điền TODO).** Hoàn thành `integer_break_todo(n)` theo Cách 3 (DP bottom-up).

In [35]:
def integer_break_todo(n):
    # TODO 1: tạo bảng f có n + 1 phần tử, f[1] = 1
    # TODO 2: for i từ 2 đến n: đặt f[i] = 0
    # TODO 3: for j từ 1 đến i - 1: f[i] = max(f[i], f[i - j] * j, (i - j) * j)
    # TODO 4: trả về f[n]
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `integer_break_equal2(n)` theo Cách 4 (thử mọi số phần `k`, mỗi `k` dùng `divmod(n, k)`) **mà không nhìn lại code**. Rồi tự chứng minh vì sao các phần chênh nhau không quá `1` là tối ưu cho `k` phần.

**Bài 5 (biến thể / mini challenge).** (a) Viết `max_product_at_least_k_parts(n, k)`: tích lớn nhất khi tách `n` thành **ít nhất `k`** phần dương (với `k <= n`). Gợi ý: thử mọi số phần `m` từ `k` đến `n`. (b) Viết `integer_break_parts_todo(n)` trả về **một cách tách tối ưu** (danh sách các phần).

In [36]:
def max_product_at_least_k_parts_todo(n, k):
    # TODO: thử mọi m từ k đến n, dùng công thức chia đều cho đúng m phần, lấy max
    pass


def integer_break_parts_todo(n):
    # TODO: trả về danh sách các phần của một cách tách tối ưu (có thể dùng công thức số 3)
    pass

# KHÔNG gọi hàm TODO ở đây.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `divmod` cho `(thương, dư)`. `integerBreak(6)` là `3 + 3`. Bài 3: nhớ `f` có `n + 1` ô và vòng `j` chạy `range(1, i)`. Bài 5a: nếu `m` phần thì mỗi phần khoảng `n / m`.
- **Mức 2 (vừa):** Bài 2: `f = [-, 1, 1, 2, 4, 6, 9, 12]`; công thức: `7 // 3 = 2` dư `1` ⇒ `3^1 · 4 = 12`. Bài 4: với `k` phần, `q, r = divmod(n, k)`, có `r` phần `q + 1`. Bài 5b: dư `0/1/2` ứng với toàn `3`, thêm `2, 2`, thêm `2`.
- **Mức 3 (sâu):** Bài 3: `f[i] = max(f[i], f[i - j] * j, (i - j) * j)`. Bài 4 (chứng minh): nếu `a >= b + 2` thì `(a - 1)(b + 1) - ab = a - b - 1 >= 1 > 0`. Bài 5a: `max(best_exact_k(n, m) for m in range(k, n + 1))`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [37]:
# Bài 1
assert divmod(17, 5) == (3, 2)
assert max(4, 2 * 3, 1 * 5) == 6
assert 3 ** 3 * 2 == 54
assert Solution().integerBreak(6) == 9
assert Solution().integerBreak(11) == 54

# Bài 2
table = [0, 1, 1, 2, 4, 6, 9, 12]
assert [Solution().integerBreak(n) for n in range(2, 8)] == table[2:]
assert table[7] == 3 ** 1 * 4

# Bài 3
def integer_break_todo_ans(n):
    f = [1] * (n + 1)
    for i in range(2, n + 1):
        f[i] = 0
        for j in range(1, i):
            f[i] = max(f[i], f[i - j] * j, (i - j) * j)
    return f[n]

for n in range(2, 59):
    assert integer_break_todo_ans(n) == Solution().integerBreak(n)

# Bài 4
def integer_break_equal2(n):
    return max((n // k + 1) ** (n % k) * (n // k) ** (k - n % k) for k in range(2, n + 1))

for n in range(2, 59):
    assert integer_break_equal2(n) == Solution().integerBreak(n)

# Bài 5a: tích lớn nhất khi tách thành ÍT NHẤT k phần
def max_product_at_least_k_parts(n, k):
    return max(best_exact_k(n, m) for m in range(k, n + 1))

assert max_product_at_least_k_parts(10, 2) == 36
assert max_product_at_least_k_parts(10, 4) == 3 * 3 * 2 * 2     # ít nhất 4 phần: 3+3+2+2 = 36
assert max_product_at_least_k_parts(10, 10) == 1
# đối chiếu vét cạn trên n nhỏ
for n in range(2, 16):
    for k in range(1, n + 1):
        brute = max(math.prod(p) for p in gen(n, n) if len(p) >= k)
        assert max_product_at_least_k_parts(n, k) == brute, (n, k)

# Bài 5b
def integer_break_parts_ans(n):
    return integer_break_parts(n)[1]

for n in range(2, 59):
    parts = integer_break_parts_ans(n)
    assert sum(parts) == n and len(parts) >= 2 and math.prod(parts) == Solution().integerBreak(n)
print("Tất cả đáp án bài tập đều pass")

Tất cả đáp án bài tập đều pass


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (công thức số `3`)

```python
class Solution:
    def integerBreak(self, n: int) -> int:
        if n < 4:
            return n - 1
        q, r = divmod(n, 3)
        if r == 0:
            return 3 ** q
        if r == 1:
            return 3 ** (q - 1) * 4
        return 3 ** q * 2
```

**Template DP** (khi không muốn nhớ công thức)

```python
class Solution:
    def integerBreak(self, n: int) -> int:
        f = [1] * (n + 1)
        for i in range(2, n + 1):
            f[i] = 0
            for j in range(1, i):
                f[i] = max(f[i], f[i - j] * j, (i - j) * j)
        return f[n]
```

**Checklist**

- [ ] Đã nhớ bắt buộc tách thành **ít nhất hai** phần (không giữ nguyên `n`)?
- [ ] Đã xử lý riêng `n = 2` (đáp án `1`) và `n = 3` (đáp án `2`)?
- [ ] Dư `1` khi chia `3` đã đổi thành `2 + 2` (nhân `4`) thay vì `3 + 1`?
- [ ] Trong DP có xét **cả hai** khả năng: giữ nguyên `i - j` và tách tiếp `f[i - j]`?
- [ ] Đã thử `n = 4` (đáp án `4`), `n = 10` (đáp án `36`), `n = 58` (đáp án `1549681956`)?
- [ ] Skeleton đúng `class Solution` / `def integerBreak(self, n: int) -> int`?